In [2]:
import time
import tracemalloc
from datasketch import MinHash
from pathlib import Path


def resolve_data_file(filename: str) -> Path:
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "datasets" / filename,
        Path.cwd().parent / "datasets" / filename,
        Path.cwd() / "notebooks" / "datasets" / filename,
        Path.cwd().parent / "notebooks" / "datasets" / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Impossible de trouver {filename} dans les emplacements attendus.")


def get_shingles(text: str, k: int = 2) -> set:
    words = text.lower().split()
    return set(" ".join(words[i:i+k]) for i in range(len(words) - k + 1))


def minhash_similarity(text1: str, text2: str, num_perm: int = 128) -> float:
    m1, m2 = MinHash(num_perm=num_perm), MinHash(num_perm=num_perm)
    for shingle in get_shingles(text1):
        m1.update(shingle.encode("utf8"))
    for shingle in get_shingles(text2):
        m2.update(shingle.encode("utf8"))
    return m1.jaccard(m2)


# --- Chargement de tous les documents du corpus ---
fichiers = {
    "original": "doc_original.txt",
    "paraphrase": "doc_paraphrase.txt",
    "identique": "doc_identique.txt",
    "different_1": "doc_different_1.txt",
    "different_2": "doc_different_2.txt",
}
docs = {nom: open(resolve_data_file(f), encoding="utf-8").read() for nom, f in fichiers.items()}

# --- Vérité terrain ---
verite_terrain = {
    ("original", "identique"): 1,
    ("original", "paraphrase"): 1,
    ("original", "different_1"): 0,
    ("original", "different_2"): 0,
    ("paraphrase", "different_1"): 0,
    ("paraphrase", "different_2"): 0,
    ("different_1", "different_2"): 0,
}

SEUIL = 0.5

# --- Évaluation de MinHash sur tout le corpus ---
tracemalloc.start()
start = time.time()

vp = fp = vn = fn = 0
details = []
for (a, b), label in verite_terrain.items():
    score = minhash_similarity(docs[a], docs[b])
    prediction = 1 if score >= SEUIL else 0
    details.append((a, b, score, label, prediction))
    if prediction == 1 and label == 1: vp += 1
    elif prediction == 1 and label == 0: fp += 1
    elif prediction == 0 and label == 0: vn += 1
    elif prediction == 0 and label == 1: fn += 1

duree = time.time() - start
_, memoire_pic = tracemalloc.get_traced_memory()
tracemalloc.stop()

precision = vp / (vp + fp) if (vp + fp) > 0 else 0
rappel = vp / (vp + fn) if (vp + fn) > 0 else 0

# --- Affichage détaillé ---
print("Détail des paires :")
for a, b, score, label, prediction in details:
    statut = "✅" if prediction == label else "❌"
    print(f"  {a} vs {b} : score={score:.2%} | attendu={label} | prédit={prediction} {statut}")

print(f"\nRésultats MinHash :")
print(f"  Précision : {precision:.2%}")
print(f"  Rappel    : {rappel:.2%}")
print(f"  Temps de calcul (7 paires) : {duree:.4f}s")
print(f"  Mémoire (pic) : {memoire_pic / 1024:.2f} Ko")

Détail des paires :
  original vs identique : score=100.00% | attendu=1 | prédit=1 ✅
  original vs paraphrase : score=3.12% | attendu=1 | prédit=0 ❌
  original vs different_1 : score=10.94% | attendu=0 | prédit=0 ✅
  original vs different_2 : score=0.00% | attendu=0 | prédit=0 ✅
  paraphrase vs different_1 : score=1.56% | attendu=0 | prédit=0 ✅
  paraphrase vs different_2 : score=0.78% | attendu=0 | prédit=0 ✅
  different_1 vs different_2 : score=1.56% | attendu=0 | prédit=0 ✅

Résultats MinHash :
  Précision : 100.00%
  Rappel    : 50.00%
  Temps de calcul (7 paires) : 0.3296s
  Mémoire (pic) : 49.24 Ko
